In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
--CallCheck
--SELECT * FROM dataengineerflightproject.silver.silver_passengers

-- Dimension Surrogate Key
-- Make sure that the new (incremented) row will get a unique surrogate key



## Parameters 
- What is required?

In [0]:

#Key Cols List
key_cols = "['flight_id']"
key_cols_list = eval(key_cols)

#CDC Column
cdc_col = "modified_date"

#Backdated Refresh
backdated_refresh = ""

#Catalog
catalog = "dataengineerflightproject"

##Source##
source_object = "silver_flights"
source_schema = "silver"

###Target Schema#
target_object = "Dim_flights"
target_schema = "gold"

#Surrogate Key
surrogate_key = "dim_flights_key"

In [0]:

# #Key Cols List
# key_cols = "['airport_id']"
# key_cols_list = eval(key_cols)

# #CDC Column
# cdc_col = "modified_date"

# #Backdated Refresh
# backdated_refresh = ""

# #Catalog
# catalog = "dataengineerflightproject"

# ##Source##
# source_object = "silver_airports"
# source_schema = "silver"

# ###Target Schema#
# target_object = "Dim_airports"
# target_schema = "gold"

# #Surrogate Key
# surrogate_key = "dim_airports_key"

In [0]:
# #Key Cols List
# key_cols = "['passenger_id']"
# key_cols_list = eval(key_cols)

# #CDC Column
# cdc_col = "modified_date"

# #Backdated Refresh
# backdated_refresh = ""

# #Catalog
# catalog = "dataengineerflightproject"

# ##Source##
# source_object = "silver_passengers"
# source_schema = "silver"

# ###Target Schema#
# target_object = "Dim_passengers"
# target_schema = "gold"

# #Surrogate Key
# surrogate_key = "dim_passengers_key"

## Incremental Data Ingestion

### Last Load Date

In [0]:
##Check the last load date with conditional statment
if len(backdated_refresh) == 0: ## No backdated refresh

    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"): #But already have a table at gold
       last_load_date = spark.sql(f"SELECT max({cdc_col}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
       #Pull a latest timestamp from the target table

    else: ##No table at gold
       last_load_date = "1900-01-01 00:00:00"

else: ##Add variable last_load_date as the same as the original
    last_load_date = backdated_refresh

#Test
last_load_date


In [0]:
df_src = spark.sql(f"SELECT * FROM {catalog}.{source_schema}.{source_object} WHERE {cdc_col} > '{last_load_date}'")


### Detect Old Record and New Record

### Tackle with initial load

In [0]:
#Key Column String
key_cols_string = ", ".join(key_cols_list)

In [0]:
#--> Prepare the Gold Layer table to ready to join in any case

#If the table at the gold is exist
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    #Key Column String for incremetal
    key_cols_string_incremental = ", ".join(key_cols_list)
    df_tgt = spark.sql(f"SELECT {key_cols_string_incremental}, {surrogate_key}, create_date, update_date FROM {catalog}.{target_schema}.{target_object}")
#if not
else: #Create Psuedo Table (Empty Table) 
    #Need to have connected column
    #have surrogate key

    #To set the missing column with null value to make the column exist (Initial)
    key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
    key_cols_string_init = ", ".join(key_cols_string_init)

    df_tgt = spark.sql(f"SELECT {key_cols_string_init}, CAST('0' AS INTEGER) AS {surrogate_key}, CAST('1900-01-01 00:00:00' AS timestamp) AS create_date, CAST('1900-01-01 00:00:00' AS timestamp) AS update_date WHERE 1=0")

    #--> Meaning this is a initial table


In [0]:
df_tgt.display()

In [0]:
#key_cols_string = ' , '.join(key_cols_list)
#spark.sql(f"SELECT {key_cols_string}, {surrogate_key}, create_date, update_date FROM {catalog}.{target_schema}.{target_object}")

In [0]:
#In case of not have any {key_cols_string} because is empty
# '' AS flight_id, '' AS airline, '' AS origin, '' AS destination,....

#List Comprehension (Built a set of the data through list
#Because need to build the pesudo table with a need criteria from key_cols_list
#Built it with list comprehension with null to have the empty space

key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
#--> ["'' AS ['flight_id']", "'' AS ['test_1']"]
key_cols_string_init = ", ".join(key_cols_string_init)
#--> "'' AS ['flight_id'], '' AS ['test_1']"
key_cols_string_init


### Join Condition

In [0]:
#Join Condition

join_condition = ' AND '.join([f"src.{i} = tgt.{i}" for i in key_cols_list])

In [0]:
df_src.createOrReplaceTempView("src")
df_tgt.createOrReplaceTempView("tgt")

df_join = spark.sql(f"""
            SELECT src.*, 
                tgt.{surrogate_key}, 
                tgt.create_date, 
                tgt.update_date
            FROM src
            LEFT JOIN tgt
            ON {join_condition}
            """)

In [0]:
df_join.display()

In [0]:
###Seperate Between Old and New Record by using the {surrogate_key} filter

df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
df_new = df_join.filter(col(f'{surrogate_key}').isNull())

df_old.display()
df_new.display()

# Enriching DFS

## Preparing df_old

In [0]:
df_old_enr = df_old.withColumn('update_date', lit(current_timestamp()))


## Preparing df_new

In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    #Find Maximum Surrogate Key
    max_surrogate_key = spark.sql(f"""
                            SELECT max({surrogate_key}) FROM {catalog}.{target_schema}.{target_object}
                                """).collect()[0][0]
    
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())\
        .withColumn('create_date', current_timestamp())\
        .withColumn('update_date', current_timestamp())

#Create New one
else:
    max_surrogate_key = 0 #Start from 0 + 1
    df_new_enr = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key) + lit(1) + monotonically_increasing_id())\
        .withColumn('create_date', current_timestamp())\
        .withColumn('update_date', current_timestamp())

#Write to Delta Lake
#df_new.write.mode("append").format("delta").saveAsTable(f"{catalog}.{target_schema}.{target_object}")

## Union old and new records

In [0]:
df_union = df_old_enr.unionByName(df_new_enr)
df_union.display()

## Upsert Table

Upsert on flight_id DimFilght Key and 

In [0]:
from delta.tables import DeltaTable

In [0]:
#Insert data if table is not exist 
#Upsert if the table is exist
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    #df_union.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{target_schema}.{target_object}")
    dlt_object = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_object.alias("tgt").merge(df_union.alias("src"), f"tgt.{surrogate_key} = src.{surrogate_key}")\
        .whenMatchedUpdateAll(condition= f"src.{cdc_col} >= tgt.{cdc_col}")\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_union.write.format("delta").mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")
        

In [0]:
%sql
SELECT * FROM dataengineerflightproject.gold.dim_passengers